In [ ]:
%pip install langchain langchain-huggingface langchain-community sentence-transformers
%pip install databricks-langchain
dbutils.library.restartPython()

In [ ]:
%pip install itables
dbutils.library.restartPython()

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# ============================================
# 1. GTE Model
# ============================================
gte_embeddings = HuggingFaceEmbeddings(
    model_name="thenlper/gte-large",
    model_kwargs={'device': 'cpu'},  # Use 'cuda' for GPU
    encode_kwargs={'normalize_embeddings': True}
)

# ============================================
# 2. BGE Model
# ============================================
bge_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# ============================================
# 3. MiniLM Model
# ============================================
minilm_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print("✓ All embedding models loaded!")


In [ ]:
# Sample documents to embed
documents = [
    "Machine learning is a subset of artificial intelligence",
    "Embeddings are numerical representations of text",
    "Vector databases store and search embeddings efficiently",
    "Deep learning uses neural networks with multiple layers",
    "Natural language processing enables computers to understand text",
    "Transformers revolutionized NLP with attention mechanisms",
    "Semantic search finds similar documents based on meaning",
    "Retrieval augmented generation combines search with generation",
]

# Create document IDs
doc_ids = list(range(1, len(documents) + 1))

In [ ]:
import pandas as pd
from datetime import datetime

# Generate embeddings for all documents using each model
print("Generating GTE embeddings...")
gte_vectors = gte_embeddings.embed_documents(documents)

print("Generating BGE embeddings...")
bge_vectors = bge_embeddings.embed_documents(documents)

print("Generating MiniLM embeddings...")
minilm_vectors = minilm_embeddings.embed_documents(documents)

print("✓ All embeddings generated!")

# Print embedding dimensions
print(f"\nEmbedding Dimensions:")
print(f"  GTE: {len(gte_vectors[0])}")
print(f"  BGE: {len(bge_vectors[0])}")
print(f"  MiniLM: {len(minilm_vectors[0])}")


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, FloatType, TimestampType
from pyspark.sql.functions import current_timestamp, lit

# ============================================
# Option A: Unified Table with All Embeddings
# ============================================

# Create Pandas DataFrame with all embeddings
df_pandas = pd.DataFrame({
    'id': doc_ids,
    'text': documents,
    'gte_embedding': gte_vectors,
    'bge_embedding': bge_vectors,
    'minilm_embedding': minilm_vectors,
})

# Define schema for Spark DataFrame
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("text", StringType(), True),
    StructField("gte_embedding", ArrayType(FloatType()), True),
    StructField("bge_embedding", ArrayType(FloatType()), True),
    StructField("minilm_embedding", ArrayType(FloatType()), True),
])

# Convert to Spark DataFrame
df_spark = spark.createDataFrame(df_pandas, schema=schema)
df_spark = df_spark.withColumn("created_at", current_timestamp())

# Write to Delta table
df_spark.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("users.anirvan_sen.embeddings_all_models")

print("✓ Unified embeddings table saved!")
display(spark.table("users.anirvan_sen.embeddings_all_models"))



#### Now we have created three index  gte_embedding_vs_index , bge_embedding_vs_index , minilm_embedding_vs_index from catalog explorer ui

https://docs.databricks.com/aws/en/vector-search/create-vector-search#create-index-using-the-ui


### Evaluate Different embedding model

In [ ]:
import mlflow
import pandas as pd
from mlflow.entities import Document
from mlflow.genai.scorers import RetrievalRelevance, RetrievalGroundedness
from databricks.vector_search.client import VectorSearchClient
from langchain_huggingface import HuggingFaceEmbeddings
from typing import List

# ============================================
# SETUP
# ============================================
client = VectorSearchClient()
username = spark.sql("SELECT current_user()").first()[0]
experiment_path = f"/Users/{username}/embedding_model_comparison_v3"
mlflow.set_experiment(experiment_path)

# ============================================
# DIVERSE EVALUATION QUERIES
# ============================================
eval_dataset = [
    {"inputs": {"query": "How do computers learn from data?"}},
    {"inputs": {"query": "Converting words into numbers"}},
    {"inputs": {"query": "Neural network architectures"}},
    {"inputs": {"query": "Text understanding by machines"}},
    {"inputs": {"query": "AI models that process sequences"}},
    {"inputs": {"query": "Finding similar content"}},
    {"inputs": {"query": "Attention mechanism in neural networks"}},
    {"inputs": {"query": "Dense vector representations"}},
    {"inputs": {"query": "Combining search results with language models"}},
    {"inputs": {"query": "Storing and querying high-dimensional data"}},
    {"inputs": {"query": "How to train a model on custom data?"}},
    {"inputs": {"query": "Difference between supervised and unsupervised learning"}},
    {"inputs": {"query": "vectors"}},
    {"inputs": {"query": "attention"}},
    {"inputs": {"query": "encoding text"}},
]

# ============================================
# INITIALIZE EMBEDDING MODELS
# ============================================
gte_model = HuggingFaceEmbeddings(model_name="thenlper/gte-large", model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True})
bge_model = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5", model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True})
minilm_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True})

models = {
    "GTE": {"index_name": "users.anirvan_sen.gte_embedding_vs_index", "embedding_model": gte_model},
    "BGE": {"index_name": "users.anirvan_sen.bge_embedding_vs_index", "embedding_model": bge_model},
    "MiniLM": {"index_name": "users.anirvan_sen.minilm_embedding_vs_index", "embedding_model": minilm_model}
}

# ============================================
# CREATE RETRIEVER & RAG FACTORY
# ============================================
def create_retriever(index_name: str, model_name: str, embedding_model):
    @mlflow.trace(span_type="RETRIEVER")
    def retrieve(query: str) -> List[Document]:
        query_vector = embedding_model.embed_query(query)
        index = client.get_index(index_name=index_name)
        results = index.similarity_search(query_vector=query_vector, columns=["id", "text"], num_results=5)
        return [
            Document(id=str(row[0]), page_content=row[1], metadata={"model": model_name})
            for row in results.get('result', {}).get('data_array', [])
        ]
    return retrieve

def create_rag_app(retriever_fn):
    @mlflow.trace
    def rag_app(query: str) -> dict:
        docs = retriever_fn(query)
        context = "\n".join([doc.page_content for doc in docs])
        return {"response": f"Based on context: {context[:300]}..."}
    return rag_app

# ============================================
# RUN EVALUATIONS
# ============================================
print("Starting evaluations...\n")
for model_name, config in models.items():
    print(f"Evaluating {model_name}...")
    retriever = create_retriever(config["index_name"], model_name, config["embedding_model"])
    rag_app = create_rag_app(retriever)
    
    with mlflow.start_run(run_name=f"{model_name}_evaluation"):
        results = mlflow.genai.evaluate(
            data=eval_dataset,
            predict_fn=rag_app,
            scorers=[RetrievalRelevance(), RetrievalGroundedness()]
        )
        mlflow.log_param("embedding_model", model_name)
    
    print(f"✓ {model_name} done!\n")

# ============================================
# COMPARE RESULTS
# ============================================
print("\n" + "="*80)
print("MODEL COMPARISON RESULTS")
print("="*80 + "\n")

all_runs = mlflow.search_runs(experiment_names=[experiment_path])

filtered = all_runs[
    all_runs['metrics.retrieval_groundedness/mean'].notna()
    & all_runs['metrics.retrieval_relevance/mean'].notna()
    & all_runs['metrics.retrieval_relevance/precision/mean'].notna()
]


comparison_df = filtered[[
    'tags.mlflow.runName', 
    'params.embedding_model', 
    'metrics.retrieval_relevance/mean',
    'metrics.retrieval_groundedness/mean',
    'metrics.retrieval_relevance/precision/mean'
]].copy()

comparison_df.columns = ['Run Name', 'Model', 'Retrieval Relevance', 'Retrieval Groundedness', 'Precision']
comparison_df = comparison_df.round(4)

display(comparison_df)

# ============================================
# BEST PERFORMERS
# ============================================
print("\n" + "="*80)
print("BEST PERFORMERS")
print("="*80 + "\n")

best_relevance_idx = all_runs['metrics.retrieval_relevance/mean'].idxmax()
best_groundedness_idx = all_runs['metrics.retrieval_groundedness/mean'].idxmax()

best_relevance = all_runs.loc[best_relevance_idx]
best_groundedness = all_runs.loc[best_groundedness_idx]

print(f"🏆 Best Retrieval Relevance: {best_relevance['params.embedding_model']}")
print(f"   Score: {best_relevance['metrics.retrieval_relevance/mean']:.4f}\n")

print(f"🏆 Best Retrieval Groundedness: {best_groundedness['params.embedding_model']}")
print(f"   Score: {best_groundedness['metrics.retrieval_groundedness/mean']:.4f}\n")

# ============================================
# RANKING TABLE
# ============================================
print("\n" + "="*80)
print("RANKINGS")
print("="*80 + "\n")



ranking_df = filtered[['params.embedding_model', 'metrics.retrieval_relevance/mean', 'metrics.retrieval_groundedness/mean']].copy()
ranking_df.columns = ['Model', 'Retrieval Relevance', 'Retrieval Groundedness']

print("By Retrieval Relevance:")
display(ranking_df.sort_values('Retrieval Relevance', ascending=False).round(4).reset_index(drop=True))

print("\nBy Retrieval Groundedness:")
display(ranking_df.sort_values('Retrieval Groundedness', ascending=False).round(4).reset_index(drop=True))

print("\n✅ Evaluation Complete!")


## Final Summary

* Retrieval Relevance (finding relevant documents): MiniLM and GTE are tied at 0.36 and are the best.
* Retrieval Groundedness (accurate/trustworthy responses): BGE is the best at 0.8667, while GTE and MiniLM are tied at 0.8.
* MiniLM and GTE are better at finding relevant documents, but BGE produces more accurate and trustworthy responses—choose based on whether you prioritize retrieval relevance or answer groundedness.